<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>

<p><font size="5" color='grey'> <b>
DeepAgents: Autonomes Harness-Pattern
</b></font> </br></p>

---

In [ ]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q deepagents

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M32-DeepAgents-Harness"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment, get_ipinfo, setup_api_keys,
    mprint, install_packages, mermaid, load_prompt, show_trace
)
setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

# 1 | Übersicht
---

<p><font color='black' size="5">
Von der Engine zum fertigen Fahrzeug
</font></p>

In den vorherigen Modulen haben wir mit **LangGraph** gearbeitet — der *Laufzeitumgebung* für Agenten-Graphen.
LangGraph liefert die Engine: State, Nodes, Edges, Checkpointing.
Jede Pipeline wurde von Hand zusammengebaut — Planung, Dateiverwaltung, Sub-Agenten.
Das gibt vollständige Kontrolle und ermöglicht präzises Debugging.

**DeepAgents** baut auf genau diesem Fundament auf. Es ist das offizielle **Agent-Harness**
von LangChain: eine opinionated Schicht über LangChain + LangGraph, die die häufigsten
Infrastruktur-Patterns *eingebaut* liefert.

> **Wichtig:** DeepAgents erfordert LangGraph-Kenntnisse — nicht als "nice to know",
> sondern weil die Harness bricht. Wer die LangGraph-Grundlagen (StateGraph bis Supervisor Pattern) versteht, kann debuggen.
> Wer nur die Harness kennt, steht bei Fehlern vor einer Blackbox.

LangChains eigene Begriffshierarchie:

| Begriff | Rolle | Beispiel |
|---------|-------|----------|
| **Runtime** | Laufzeitumgebung für Graphen | LangGraph |
| **Framework** | Orchestrierung, Chains, Tools | LangChain |
| **Harness** | Opinionated Agent mit Batteries Included | **DeepAgents** |


In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Ökosystem-Einordnung</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TB
    subgraph runtime ["<b>Schicht 1: Runtime</b>"]
        LG["LangGraph\nStateGraph · Checkpointing\nStreaming · Persistence"]
    end
    subgraph framework ["<b>Schicht 2: Framework</b>"]
        LC["LangChain\nChains · Tools · Agents\nRAG · LCEL"]
    end
    subgraph harness ["<b>Schicht 3: Harness</b>"]
        DA(["DeepAgents\n · Filesystem\n · Planning · Sub-Agenten · Memory"])
    end
    subgraph real ["<b>Inspirierende Systeme</b>"]
        CC["Claude Code"]
        MN["Manus"]
        DR["Deep Research"]
    end
    LG --> LC --> DA
    CC & MN & DR -.->|"gleiches Muster"| DA
    style LG fill:#1565C0,color:#fff
    style LC fill:#2E7D32,color:#fff
    style DA fill:#E65100,color:#fff
    style CC fill:#37474F,color:#fff
    style MN fill:#37474F,color:#fff
    style DR fill:#37474F,color:#fff
'''
mermaid(diagram, width=750)

# 2 | DeepAgents: Architektur & Konzepte
---

<p><font color='black' size="5">
Die drei Säulen eines Harness
</font></p>

DeepAgents liefert drei Infrastruktur-Schichten, die im Kursverlauf manuell aufgebaut wurden.
Wer diese Muster kennt, versteht was die Harness intern tut — und kann eingreifen wenn nötig:

| Säule | Was es löst | Eingebautes Feature |
|-------|-------------|--------------------|
| **Planning** | Komplexe Aufgaben strukturiert abarbeiten | `write_todos` — automatische To-do-Liste |
| **Filesystem** | Kontext-Overflow bei langen Ergebnissen | `ls`, `read_file`, `write_file`, `edit_file` |
| **Sub-Agenten** | Spezialisierte Teilaufgaben isolieren | `subagents=[...]` Parameter |

**Zusätzlich ab v0.2/v0.4:**
- Pluggable Backends (LangGraph State, LangGraph Store, lokales Filesystem)
- Sandbox-Ausführung (Runloop, Daytona, Modal) für sicheres Code-Execution
- LangSmith-Integration out-of-the-box
- `interrupt_on=` für Human-in-the-Loop (*Human-in-the-Loop*-Pattern)
- DeepAgents CLI für lokale Entwicklung

**Aktuelle Version:** 0.4.x (Stand März 2026)

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: DeepAgents Architektur</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    U(["Nutzer-Anfrage"])
    DA["create_deep_agent()"]
    subgraph s1 ["Säule 1: Planning"]
        PT["write_todos\nAufgaben strukturieren\nFortschritt tracken"]
    end
    subgraph s2 ["Säule 2: Filesystem"]
        FST["ls · read_file\nwrite_file · edit_file\nKontext-Overflow vermeiden"]
    end
    subgraph s3 ["Säule 3: Sub-Agenten"]
        SA["Sub-Agent A"]
        SB["Sub-Agent B"]
    end
    subgraph infra ["LangGraph Runtime"]
        CP["Checkpointing"]
        ST["Streaming"]
        LS["LangSmith"]
    end
    U --> DA
    DA --> PT & FST & SA & SB
    DA --> CP & ST & LS

    style U   fill:#E65100,color:#fff
    style DA  fill:#1565C0,color:#fff
    style PT  fill:#2E7D32,color:#fff
    style FST fill:#4A148C,color:#fff
    style SA  fill:#37474F,color:#fff
    style SB  fill:#37474F,color:#fff
'''
mermaid(diagram, width=1200)

<p><font color='black' size="5">
Fundament vs. Harness: Kontrolle vs. Schnelles Deployment
</font></p>

Im bisherigen Kursverlauf haben wir alle Komponenten manuell gebaut — nicht als Übung, sondern weil
dieses Verständnis drei konkrete Vorteile bringt:

- **Debugging:** Wenn die Harness bricht, muss man verstehen was darunter passiert
- **Transparenz:** Jeder Node, jede Edge, jede Routing-Entscheidung ist nachvollziehbar
- **Erweiterbarkeit:** Custom-Anforderungen, die die Harness nicht abdeckt

DeepAgents kapselt dieselben LangGraph-Mechanismen — weniger Code, dafür weniger
direkte Kontrolle. Das ist kein Widerspruch: Es ist die richtige Schicht für den
richtigen Kontext.

```python
# LangGraph manuell (M13-Pattern): ~100 Zeilen — vollständige Kontrolle
class MyState(TypedDict):
    todos: list
    messages: Annotated[list, add_messages]

def planning_node(state): ...   # write_todos selbst bauen
def fs_node(state): ...         # Datei-Tools selbst bauen
def subagent_node(state): ...   # Sub-Agent-Loop selbst bauen

builder = StateGraph(MyState)
builder.add_node("planning", planning_node)
# ... weitere 60 Zeilen

# DeepAgents Harness: 4 Zeilen — schnelles Deployment
from deepagents import create_deep_agent

agent = create_deep_agent(
    tools=[my_tool],
    system_prompt="Du bist ein hilfreicher Agent.",
)
# Planning, Filesystem, Sub-Agenten: automatisch eingebaut
# Was darunter passiert: LangGraph — wie in M13–M31 gelernt
```

# 3 | Hands-On: DeepAgents in Aktion
---

<p><font color='black' size="5">
Schritt 1: Einfachster Deep Agent
</font></p>

`create_deep_agent()` braucht keine Pflicht-Parameter — er kommt mit sinnvollen Defaults.
Automatisch eingebaut: Planning-Tool (`write_todos`), Filesystem-Tools, Standard-Prompt.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Minimaler Deep Agent</font> </br></p>

import time
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool

# ── Konfigurationskonstante ────────────────────────────────────────────────
# recursion_limit: Jeder Sub-Schritt des Agenten zählt separat.
# write_todos + pro Thema: LLM-Call + Update ≈ 8 Schritte pro einfacher Aufgabe.
# Komplexe Aufgaben oder Sub-Agenten benötigen höhere Limits (c039: 50, Planning: 100).
MAX_RECURSION = 30   # Basis-Limit für einfache Agent-Läufe

# System-Prompt: Filesystem-first-Tendenz unterdrücken
# DeepAgents greift bei Wissensfragen standardmäßig zuerst auf grep/Filesystem zu.
# Für Demo-Zwecke: Agent soll direkt aus Modell-Wissen antworten.
minimal_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4o-mini", temperature=0.0),
    system_prompt=(
        "Du bist ein hilfreicher Assistent. "
        "Beantworte Fragen direkt aus deinem Wissen — "
        "ohne Filesystem-Suche, sofern nicht explizit nach Dateien gefragt wird."
    ),
)

mprint(f'✅ Minimaler DeepAgent erstellt — Planning + Filesystem automatisch eingebaut')
mprint(f'   MAX_RECURSION = {MAX_RECURSION} (Basis-Limit für einfache Läufe)')

In [ ]:
#@markdown   <p><font size="4" color='green'>  Interner Graph: Minimaler DeepAgent</font> </br></p>

from IPython.display import Image, display

# Hauptgraph als PNG
print("── Hauptgraph ──")
display(Image(minimal_agent.get_graph(xray=True).draw_mermaid_png()))


**Befund**:

Sub-Agenten erscheinen nicht als LangGraph-Subgraphen im Graphen.
get_subgraphs() liefert keine Einträge.


DeepAgents startet automatisch (spawnt) Sub-Agenten zur Laufzeit als Tool-Calls —
nicht zur Compile-Zeit als registrierte CompiledStateGraph-Nodes.
Der Koordinator-Graph sieht aus wie ein minimaler Agent;
die Sub-Agenten existieren erst wenn der Agent sie aufruft.


Das ist ein wesentlicher Unterschied zum *Hierarchical Agent Teams*-Pattern,
wo jeder Sub-Agent als eigener StateGraph kompiliert und
statisch im Hauptgraph registriert wird.



In [ ]:
#@markdown   <p><font size="4" color='green'>  Planning Demo: write_todos in Aktion</font> </br></p>

# Aufgabe: kurze, gebundene Antworten → write_todos feuert, aber jeder Schritt bleibt schnell
aufgabe = (
    "Beantworte in je einem Satz:\n"
    "1. Was ist LangGraph?\n"
    "2. Was ist LangSmith?\n"
    "Plane deine Schritte zuerst."
)

print(f"Aufgabe: {aufgabe}")
print()

# DeepAgents-Schrittschätzung:
#   write_todos (1) + pro Thema: LLM-Call + Todo-Update (2×2) + Synthese (1) ≈ 8 Schritte
#   Jeder Subgraph-Schritt zählt separat → recursion_limit 100 als sicherer Puffer.
t0 = time.perf_counter()
result = minimal_agent.invoke(
    {"messages": [{"role": "user", "content": aufgabe}]},
    config={
        "recursion_limit": 100,
        "run_name": "m32-planning-demo",
        "tags": ["m32", "planning"],
    }
)
latenz = round((time.perf_counter() - t0) * 1000)

zeilen = [
    "## Planning-Demo Ergebnis", "",
    "| Metrik | Wert |",
    "|--------|------|",
    f"| Nachrichten gesamt | `{len(result['messages'])}` |",
    f"| Latenz | `{latenz} ms` |",
    "",
    "> Der Agent hat `write_todos` automatisch genutzt, um die Aufgabe in",
    "> Teilschritte zu zerlegen — ohne explizite Aufforderung.",
    "",
    "> ⚠️ **Latenz-Tipp:** Kurze, gebundene Aufgaben halten die Demo-Latenz unter 15 s.",
    "> Lange Ausgaben (z.B. 'strukturierter Überblick') multiplizieren die Token-Menge",
    "> pro Schritt und können 50+ s erzeugen — auch mit `gpt-4o-mini`.",
]
mprint("\n".join(zeilen))
print()
mprint(f"## Antwort\n\n{result['messages'][-1].content}")

In [ ]:
#@markdown   <p><font size="4" color='green'>  Trace-Analyse: Planning-Demo</font> </br></p>

import time as _time
_time.sleep(2)  # kurz warten bis Run in LangSmith verfügbar

# show_steps=True: zeigt alle Tool-Calls des letzten Runs
# → macht Filesystem-first-Tendenz (grep-Calls) direkt sichtbar
show_trace("M32-DeepAgents-Harness", limit=3, show_steps=True)

<p><font color='black' size="4">
💡 Beobachtungen zur Planning-Demo
</font></p>

**Filesystem-first-Tendenz**

DeepAgents hat bei Wissensfragen eine eingebaute Bias: Der Agent ruft zuerst `grep`
auf dem virtuellen Filesystem auf — bevor er aus Modell-Wissen antwortet.
In LangSmith sichtbar als Sequenz von `grep`-Tool-Calls mit "No matches found".

Das ist für Long-Running-Tasks sinnvoll (persistierter Kontext), für einfache
Wissensfragen kontraproduktiv: unnötige Tool-Calls, höhere Latenz.

**Gegenmittel:** System-Prompt mit expliziter Anweisung:
```python
system_prompt=(
    "Beantworte Fragen direkt aus deinem Wissen — "
    "ohne Filesystem-Suche, sofern nicht explizit nach Dateien gefragt wird."
)
```

---

**Filesystem: Kein Schreiben auf das OS-Filesystem**

Der Agent meldet gelegentlich "Datei gespeichert unter `/datei.txt`" — auf der
Colab-VM nicht auffindbar. Das ist kein Fehler:

DeepAgents' `write_file` schreibt in ein **agenten-internes, virtuelles Filesystem** —
ausschließlich für Context-Management, nicht für benutzerzugängliche Artefakte.

> Wer echte Dateien als Output braucht, schreibt ein eigenes `@tool` mit `open(..., 'w')`.

---

**Latenz: Warum Planning-Tasks langsam sind**

Jeder `write_todos`-Loop erzeugt mehrere serielle LLM-Calls. Zusätzlich:
unnötige `grep`-Calls (Filesystem-first-Tendenz) erhöhen die Latenz weiter.

| Aufgabentyp | erwartete Latenz |
|-------------|-----------------|
| Kurze Antworten + System-Prompt-Fix | 10–20 s |
| Strukturierte Übersichten | 40–80 s |
| Vergleiche mit mehreren Themen | 60–120 s |

<p><font color='black' size="5">
Schritt 2: Eigene Tools hinzufügen
</font></p>

DeepAgents ist erweiterbar — eigene `@tool`-Funktionen werden einfach übergeben.
Der Agent kombiniert die eingebauten Harness-Tools mit den eigenen.

In [ ]:
#@markdown   <p><font size="4" color='green'>  Eigene Tools definieren</font> </br></p>

@tool
def ki_glossar(begriff: str) -> str:
    """Schlägt einen KI-Begriff im Glossar nach.

    Args:
        begriff: Der nachzuschlagende Begriff (z.B. LangGraph, RAG, HITL)
    """
    glossar = {
        "langgraph":   "LangGraph: Zustandsbasierte Multi-Agent-Graphen auf Basis von LangChain. StateGraph, Checkpointing, Streaming.",
        "langchain":   "LangChain: Framework für LLM-Anwendungen. Chains, Agents, LCEL, Tools, RAG.",
        "langsmith":   "LangSmith: Tracing, Evaluation und Monitoring für LLM-Pipelines.",
        "rag":         "RAG (Retrieval-Augmented Generation): Kombination von Vektordatenbank-Suche und LLM-Synthese.",
        "hitl":        "HITL (Human-in-the-Loop): Menschliche Freigabe bei kritischen Agenten-Schritten. LangGraph: interrupt().",
        "harness":     "Harness: Unterstützendes Rahmengerüst für einen Agenten — liefert Planning, Filesystem, Sub-Agenten als Infrastruktur.",
        "deepagents":  "DeepAgents: Offizielles Agent-Harness von LangChain. Batteries Included: Planning, Filesystem, Sub-Agenten, Memory.",
    }
    key = begriff.lower().strip()
    return glossar.get(key, f'Begriff "{begriff}" nicht im Glossar. Verfügbar: {", ".join(glossar.keys())}')

@tool
def kurs_modul_info(modul_nr: str) -> str:
    """Gibt Informationen zu einem Kursmodul zurück.

    Args:
        modul_nr: Modul-Nummer (z.B. M13, M20, M32)
    """
    module = {
        "M13": "StateGraph Basics: State, Nodes, Edges, compile()",
        "M14": "Conditional Routing und Tool-Loop",
        "M18": "Human-in-the-Loop: interrupt(), Approval-Pattern",
        "M19": "Multi-Agent Patterns: Supervisor, Hierarchical, Collaborative",
        "M20": "Supervisor Pattern: Worker-Agents, Supervisor-Logik, Graph",
        "M30": "Hierarchical Agent Teams: 3-Ebenen, Tool-Delegation, asyncio",
        "M31": "Collaborative Multi-Agent (Capstone): End-to-End Research-Report",
        "M32": "DeepAgents: Autonomes Harness-Pattern (dieses Modul)",
    }
    key = modul_nr.upper().strip()
    return module.get(key, f'Modul "{modul_nr}" nicht gefunden. Verfügbar: {", ".join(module.keys())}')

# System-Prompt: Filesystem-first-Tendenz unterdrücken
custom_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4o-mini", temperature=0.0),
    tools=[ki_glossar, kurs_modul_info],
    system_prompt=(
        "Du bist Kurs-Assistent für den Agenten-Kurs. "
        "Nutze ki_glossar für Begriffserklärungen und "
        "kurs_modul_info für Modul-Informationen. "
        "Beantworte Fragen direkt über deine Tools — "
        "ohne Filesystem-Suche, sofern nicht explizit nach Dateien gefragt wird. "
        "Antworte immer auf Deutsch."
    ),
)

mprint('✅ Custom DeepAgent erstellt — eigene Tools + Harness-Tools kombiniert')

In [ ]:
#@markdown   <p><font size="4" color='green'>  Interner Graph: DeepAgent mit eigenen Tools</font> </br></p>

print("── Hauptgraph ──")
display(Image(custom_agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
#@markdown   <p><font size="4" color='green'>  Custom Agent starten</font> </br></p>

frage = (
    "Erkläre mir in 3 Schritten: Was ist DeepAgents, wozu dient ein Harness "
    "und in welchem Modul lernt man Hierarchical Agent Teams?"
)

print(f"Frage: {frage}")
print()

t0 = time.perf_counter()
result2 = custom_agent.invoke(
    {"messages": [{"role": "user", "content": frage}]},
    config={
        "recursion_limit": MAX_RECURSION,   # min. 30 für Fragen mit mehreren Tool-Calls
        "run_name": "m32-custom-tools",
        "tags": ["m32", "custom-tools"],
    }
)
latenz2 = round((time.perf_counter() - t0) * 1000)

mprint(f"**Latenz:** `{latenz2} ms`")
print()
mprint(f"## Antwort\n\n{result2['messages'][-1].content}")

<p><font color='black' size="5">
Schritt 3: Sub-Agenten spawnen
</font></p>

Der leistungsstärkste Aspekt von DeepAgents: **Sub-Agenten**.
Der Haupt-Agent kann spezialisierte Sub-Agenten für Teilaufgaben spawnen.
Jeder Sub-Agent bekommt eine eigene, isolierte Message-History —
kein Kontext-Overflow im Haupt-Agenten.

```python
# ✅ Richtig: Sub-Agent als Dict definieren
sub_researcher = {
    "name": "recherche-spezialist",
    "description": "Schlägt KI-Begriffe im Glossar nach",
    "system_prompt": "Du bist Recherche-Spezialist. Antworte auf Deutsch.",
    "tools": [ki_glossar],
}

# Haupt-Agent erhält Sub-Agenten via subagents=[...]
main_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4o-mini", temperature=0.0),
    subagents=[sub_researcher],   # ← Dict, kein CompiledStateGraph!
    system_prompt="Delegiere Recherche-Aufgaben an Sub-Agenten.",
)
```

> ⚠️ **Häufiger Fehler:** `subagents=` erwartet ein **Dict** mit den Keys
> `name`, `description`, `system_prompt`, `tools` — **nicht** das Ergebnis
> von `create_deep_agent()`. `create_deep_agent()` gibt einen `CompiledStateGraph`
> zurück, der für `subagents=` nicht verwendbar ist (`TypeError: argument of type
> 'CompiledStateGraph' is not iterable`).

**Unterschied zu *Hierarchical Agent Teams*:**

| | M30 — LangGraph manuell | DeepAgents `subagents=` |
|--|------------------------|------------------------|
| **Registrierung** | Compile-Zeit — Sub-Agent als `StateGraph` im Hauptgraph | Laufzeit — Sub-Agent wird bei Bedarf gespawnt |
| **Sichtbarkeit** | Im Graph sichtbar (`get_subgraphs()`) | Nicht im Graph sichtbar — Blackbox |
| **Kontrolle** | Volle Kontrolle über Routing und State | Harness entscheidet wann und wie gespawnt |
| **Debugging** | Jeder Sub-Agent-Schritt in LangSmith tracebar | Gespawnte Calls in LangSmith, aber kein Graph-Introspection |

In [ ]:
#@markdown   <p><font size="4" color='green'>  Sub-Agenten aufbauen</font> </br></p>

# ✅ Sub-Agenten als Dicts definieren (NICHT create_deep_agent() verwenden!)
sub_researcher = {
    "name": "recherche-spezialist",
    "description": "Schlägt KI-Begriffe im Glossar nach und gibt strukturierte Zusammenfassungen zurück",
    "system_prompt": (
        "Du bist Recherche-Spezialist. "
        "Schlage alle relevanten Begriffe im Glossar nach und "
        "gib eine strukturierte Zusammenfassung zurück. "
        "Nutze ausschließlich das ki_glossar-Tool — keine Filesystem-Suche. "
        "Antworte auf Deutsch."
    ),
    "tools": [ki_glossar],
}

sub_module_expert = {
    "name": "kurs-experte",
    "description": "Gibt Informationen zu Kursmodulen und deren Inhalten zurück",
    "system_prompt": (
        "Du bist Kurs-Experte. "
        "Gib Informationen zu Kursmodulen über das kurs_modul_info-Tool zurück — "
        "keine Filesystem-Suche. Antworte auf Deutsch."
    ),
    "tools": [kurs_modul_info],
}

# Koordinator mit Filesystem-first-Fix im System-Prompt
coordinator_agent = create_deep_agent(
    model=init_chat_model("openai:gpt-4o-mini", temperature=0.0),
    subagents=[sub_researcher, sub_module_expert],
    system_prompt=(
        "Du bist Koordinator. Zwei Sub-Agenten stehen zur Verfügung:\n"
        "- recherche-spezialist: für KI-Begriffsdefinitionen\n"
        "- kurs-experte: für Modul-Informationen\n"
        "Delegiere Aufgaben sinnvoll und kombiniere die Ergebnisse. "
        "Keine Filesystem-Suche. Antworte auf Deutsch."
    ),
)

mprint("✅ Koordinator + 2 Sub-Agenten aufgebaut (Dict-Format)")

In [ ]:
#@markdown   <p><font size="4" color='green'>  Interner Graph: Koordinator mit Sub-Agenten</font> </br></p>

print("── Hauptgraph ──")
display(Image(coordinator_agent.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
#@markdown   <p><font size="4" color='green'>  Koordinator starten</font> </br></p>

frage_komplex = (
    "Erkläre was DeepAgents und RAG sind, "
    "und nenne die Module M30 und M32 mit ihren Inhalten."
)

print(f"Frage: {frage_komplex}")
print()

t0 = time.perf_counter()
result3 = coordinator_agent.invoke(
    {"messages": [{"role": "user", "content": frage_komplex}]},
    config={
        "recursion_limit": 50,   # Sub-Agenten-Spawning braucht mehr Schritte
        "run_name": "m32-subagenten-demo",
        "tags": ["m32", "subagenten"],
    }
)
latenz3 = round((time.perf_counter() - t0) * 1000)

zeilen = [
    "## Sub-Agenten Demo Ergebnis", "",
    "| Metrik | Wert |",
    "|--------|------|",
    f"| Koordinator Messages | `{len(result3['messages'])}` |",
    f"| Latenz | `{latenz3} ms` |",
    "",
    "> Koordinator hat Aufgaben an Sub-Agenten delegiert.",
    "> Jeder Sub-Agent hatte eigene, isolierte Message-History.",
]
mprint("\n".join(zeilen))
print()
mprint(f"## Koordinator-Antwort\n\n{result3['messages'][-1].content}")

In [ ]:
#@markdown   <p><font size="4" color='green'>  Trace-Analyse: Koordinator</font> </br></p>

import time as _time
_time.sleep(2)

# Koordinator-Run: zeigt wie Sub-Agenten-Aufrufe im Trace erscheinen
show_trace("M32-DeepAgents-Harness", limit=3, show_steps=True)

# 4 | Deep Dive: Vergleich & Einordnung
---

<p><font color='black' size="5">
Wann LangGraph manuell, wann DeepAgents?
</font></p>

DeepAgents ist kein Ersatz für LangGraph — es ist eine Schicht darüber.
Wähle das richtige Werkzeug je nach Anforderung:

| Kriterium | LangGraph manuell | DeepAgents Harness |
|-----------|-------------------|--------------------|
| **Transparenz** | ✅ Vollständig (jeder Node nachvollziehbar) | ⚠️ Gekapselt (Harness entscheidet) |
| **Kontrolle** | ✅ Präzise (Custom Routing, Custom State) | ⚠️ Opinionated (Harness-Defaults) |
| **Code-Aufwand** | ⚠️ Höher (State, Nodes, Routing explizit) | ✅ Minimal (`create_deep_agent()`) |
| **Planning** | ⚠️ Selbst bauen | ✅ `write_todos` automatisch |
| **Filesystem** | ⚠️ Selbst bauen | ✅ `ls`, `read_file`, `write_file` automatisch |
| **Sub-Agenten** | ⚠️ Aufwändig (M30-Pattern) | ✅ `subagents=[...]` Parameter |
| **Debugging** | ✅ Direkt (man sieht jeden Schritt) | ⚠️ Erfordert LangGraph-Kenntnisse |
| **API-Stabilität** | ✅ LangGraph 1.0 — produktionserprobt | ⚠️ DeepAgents 0.4.x — jung, Breaking Changes möglich |
| **Use Case** | Custom Workflows, präzise Kontrolle | Autonome Long-Running Tasks |

**Faustregel:**
- ✅ LangGraph manuell: wenn jeder Schritt exakt kontrolliert werden muss
- ✅ DeepAgents: wenn ein autonomer Agent eine komplexe Aufgabe lösen soll

> ⚠️ **Wichtig:** DeepAgents setzt LangGraph-Grundkenntnisse (StateGraph bis Supervisor Pattern) voraus.
> Die Harness bricht — und dann braucht man das Fundament, um zu debuggen.
> Das Erlernen aller vorherigen Module ist keine Vorbereitung auf die Abkürzung. Es ist die Voraussetzung dafür,
> die Abkürzung sinnvoll einzusetzen.

---

**Zur API-Stabilität — eine sachliche Einschätzung:**

DeepAgents 0.4.x ist eine junge Bibliothek. Das `subagents=`-Interface wurde in diesem
Notebook bereits korrigiert: Die Schnittstelle erwartet ein **Dict**, nicht einen
`CompiledStateGraph` — ein Breaking Change, der in keiner offiziellen Migrationsanleitung
dokumentiert war.

Das ist kein Einzelfall für frühe 0.x-Releases. Konkrete Konsequenz für den produktiven Einsatz:

```python
# Empfehlung: Version pinnen, nicht "latest" verwenden
pip install deepagents==0.4.x   # konkrete Version fixieren
```

Für **experimentelle Projekte und Prototypen** ist DeepAgents gut geeignet.
Für **Production-Deployments** gilt: LangGraph direkt ist stabiler,
dokumentierter und hat ein größeres Ökosystem. DeepAgents lohnt sich zu beobachten —
ab v1.0 mit stabilem API ändert sich die Einschätzung.

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Entscheidungsbaum</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    A(["Neue Agenten-Aufgabe"])
    B{"Präzise Kontrolle\nüber jeden Schritt?"}
    C{"Autonome Aufgabe\nmit vielen Teilschritten?"}
    D["LangGraph manuell\nM13-M14-Pattern"]
    E["DeepAgents\ncreate_deep_agent()"]
    F{"Lernzweck oder\nProduction?"}
    G["LangGraph\n(Kurs M13-M31)"]
    H["DeepAgents\n(M32)"]
    A --> B
    B -->|Ja| D
    B -->|Nein| C
    C -->|Ja| E
    C -->|Nein| F
    F -->|Lernen| G
    F -->|Production| H
    style A fill:#E65100,color:#fff
    style D fill:#1565C0,color:#fff
    style E fill:#2E7D32,color:#fff
    style G fill:#37474F,color:#fff
    style H fill:#37474F,color:#fff
'''
mermaid(diagram, width=500)

<p><font color='black' size="5">
Zeitgeist: Warum jetzt?
</font></p>

2024/2025 sind autonome Agenten-Systeme mit Harness-Pattern viral gegangen.
LangChain hat in einem Blog-Post die gemeinsame Architektur analysiert:

| System | Entwickler | Planning | Filesystem | Sub-Agenten |
|--------|-----------|----------|------------|-------------|
| **Claude Code** | Anthropic | ✅ Todo-Liste | ✅ read/write/edit | ✅ Sub-Tasks |
| **Manus** | Monica | ✅ Planung | ✅ Dateiverwaltung | ✅ Delegation |
| **Deep Research** | OpenAI | ✅ Search-Plan | ✅ Context-Offloading | ⚠️ Begrenzt |
| **DeepAgents** | LangChain | ✅ `write_todos` | ✅ ls/read/write | ✅ `subagents=[]` |

> *"Applications like Deep Research, Manus, and Claude Code have gotten around the*
> *limitation of shallow agents by implementing a combination of four things:*
> *a planning tool, sub agents, access to a file system, and a detailed prompt."*
> — LangChain Blog

In [ ]:
#@markdown   <p><font size="4" color='green'>  flowchart: Kursrückblick M32</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    subgraph core ["M12-M14: LangGraph Kern"]
        SG["StateGraph · Routing"]
    end
    subgraph ma ["M19-M20: Multi-Agent"]
        SUP["Supervisor · Worker"]
    end
    subgraph sec ["M18/M25: Kontrolle"]
        HITL["HITL · Security"]
    end
    subgraph ht ["M30-M31: Hierarchisch"]
        HT["Teams · Capstone"]
    end

    subgraph m32 ["M32: DeepAgents"]
        direction LR
        SA2(["S3: Sub-Agenten"])
        FS2(["S2: Filesystem"])
        PT2(["S1: Planning"])

        SA2 -.-> FS2 -.-> PT2
    end

    %% Verbindungen zu den DeepAgents
    SG & SUP & HITL & HT --> SA2

    %% Styling
    style PT2 fill:#2E7D32,color:#fff
    style FS2 fill:#4A148C,color:#fff
    style SA2 fill:#1565C0,color:#fff
    style core fill:#f9f9f9,stroke:#333,stroke-dasharray: 5 5
'''
mermaid(diagram, width=900)

zeilen = [
    "## M32 — Kurs-Einordnung", "",
    "| Modul | Muster | In M32 |",
    "|-------|--------|--------|",
    "| M02 | `@tool` Decorator | DeepAgents-Tools werden genauso definiert |",
    "| M13 | StateGraph, Nodes | DeepAgents nutzt LangGraph intern |",
    "| M15 | Checkpointing | `checkpointer=` Parameter in `create_deep_agent()` |",
    "| M18 | Human-in-the-Loop | `interrupt_on=` Parameter |",
    "| M20 | Supervisor Pattern | DeepAgents = opinionated Supervisor |",
    "| M30 | Hierarchical Teams | `subagents=[]` als eingebautes Muster |",
    "",
    "**Kernaussage:** M12–M31 ist kein Umweg.",
    "Wer die LangGraph-Mechanismen kennt, kann DeepAgents sinnvoll einsetzen,",
    "gezielt erweitern — und debuggen wenn die Harness bricht.",
    "DeepAgents ist das produktionsreife Gerüst für Entwickler, die das Fundament kennen.",
]
mprint("\n".join(zeilen))

# A | Aufgabe
---

<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.

<p><font color='black' size="5">
DeepAgents erweitern
</font></p>

**Aufgabe 1 — Kurs-Assistent mit Filesystem-Gedächtnis:**
Baue einen DeepAgent, der Fragen zum Agenten-Kurs beantwortet und seine
Antworten automatisch in Dateien abspeichert.

Anforderungen:
- Eigenes Tool `modul_details(modul: str)` mit mindestens 5 Modulen
- System-Prompt: Agent soll Ergebnisse in `/kurs_wissen/` ablegen
- Test: Frage zu 3 verschiedenen Modulen und prüfe ob Dateien erstellt wurden

**Aufgabe 2 — Hierarchischer DeepAgent mit 2 Sub-Agenten:**
Erstelle einen Koordinator-Agent mit zwei spezialisierten Sub-Agenten:

- `sub_analyst`: Analysiert und vergleicht Konzepte (mind. 2 Tools)
- `sub_writer`: Schreibt strukturierte Zusammenfassungen (eigener System-Prompt)
- Koordinator delegiert: Recherche → Analyse → Schreiben automatisch
- Test: "Vergleiche LangGraph und DeepAgents und schreibe eine Empfehlung"

**Aufgabe 3 — DeepAgent vs. LangGraph: Direktvergleich:**
Implementiere dieselbe Aufgabe zweimal:

1. **LangGraph manuell** (*StateGraph Basics*/*Conditional Routing & Tool-Loop*-Pattern): StateGraph mit Planning-Node, Tool-Node
2. **DeepAgents Harness**: `create_deep_agent()` mit denselben Tools

Vergleiche: Zeilen Code (LOC) · Latenz (ms) · Ergebnisqualität (1–5)

> 💡 **Tipp:** Nutze `time.perf_counter()` für Latenz-Messung.